#### EDA 02: Customers table
**File:** `olist_customers_dataset.csv`  |  **Goal:** understand who the customers are and where they live, before joining them to orders.

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
customers_raw = pd.read_csv(RAW / "olist_customers_dataset.csv")
customers = customers_raw.copy()                  # all changes happen on the copy, raw stays untouched

print("shape:", customers.shape)
customers.head(3)

shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


- 99,441 rows and 5 columns: two ids (`customer_id`, `customer_unique_id`), a zip prefix, a city and a state.
- The row count equals the orders table (99,441). 

#### Data types and the zip prefix
**Question:** are the columns stored in a usable type? The zip prefix is a code, not a number, so it needs a closer look.

In [2]:
print(customers.dtypes)

# A Brazilian zip prefix has 5 digits. A number type drops a leading zero.
zip_len = customers["customer_zip_code_prefix"].astype(str).str.len().value_counts().sort_index()
print("\nzip prefix length (digits):")
print(zip_len.to_string())

short = customers["customer_zip_code_prefix"] < 10000
print("\nstates of the 4-digit prefixes:", customers.loc[short, "customer_state"].value_counts().to_dict())

# Text version with the zero restored, for display and safe text joins
customers["zip5"] = customers["customer_zip_code_prefix"].astype(str).str.zfill(5)
customers[["customer_zip_code_prefix", "zip5"]].head(3)

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

zip prefix length (digits):
customer_zip_code_prefix
4    23995
5    75446

states of the 4-digit prefixes: {'SP': 23995}


,customer_zip_code_prefix,zip5
0,14409,14409
1,9790,09790
2,1151,01151


- The zip prefix is stored as a number, so a leading zero is lost: 23,995 prefixes (24.1%) have only 4 digits.
- All of them are in SP, because SP zip codes start with 0.
- **Decision:** keep the numeric prefix (the geolocation table also stores it as a number, so the join works) and add `zip5` as a 5-digit text column for display. Joining a text zip to a numeric zip would silently match nothing.

#### Keys and duplicates
**Question:** what does one row stand for, and is any row repeated?

In [3]:
print("customer_id unique          :", customers["customer_id"].is_unique)
print("fully duplicated rows       :", customers.duplicated().sum())
print("distinct customer_unique_id :", customers["customer_unique_id"].nunique())

customer_id unique          : True
fully duplicated rows       : 0
distinct customer_unique_id : 96096


- `customer_id` is unique and no row is duplicated, so one row is one `customer_id`.
- But there are only 96,096 distinct `customer_unique_id` values. So the 99,441 rows belong to 96,096 real people.
- `customer_id` is created for every order. It is a link to one order, not a person. A person is `customer_unique_id`.

#### Missing values
**Question:** which columns have gaps?

In [4]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(customers),
                        "n_missing": customers.isna().sum(),
                        "pct_missing": (customers.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

,n_rows,n_missing,pct_missing
customer_id,99441,0,0.0
customer_unique_id,99441,0,0.0
customer_zip_code_prefix,99441,0,0.0
customer_city,99441,0,0.0
customer_state,99441,0,0.0
zip5,99441,0,0.0


- No column has a missing value.
- Nothing to impute or flag in this table.

#### People vs orders
**Question:** how many orders does a person place? Are there repeat customers?

In [5]:
orders_per_person = customers.groupby("customer_unique_id").size()

dist = orders_per_person.value_counts().sort_index().rename("n_people").to_frame()
dist.index.name = "orders_per_person"
dist["pct_people"] = (dist["n_people"] / len(orders_per_person) * 100).round(2)
display(dist)

repeat = orders_per_person[orders_per_person > 1]
print("repeat people                  :", len(repeat), f"({len(repeat) / len(orders_per_person) * 100:.1f}% of people)")
print("orders placed by repeat people :", repeat.sum(), f"({repeat.sum() / len(customers) * 100:.1f}% of orders)")

,n_people,pct_people
orders_per_person,,
1,93099,96.88
2,2745,2.86
3,203,0.21
4,30,0.03
5,8,0.01
6,6,0.01
7,3,0.00
9,1,0.00
17,1,0.00


repeat people                  : 2997 (3.1% of people)
orders placed by repeat people : 6342 (6.4% of orders)


- 93,099 people (96.9%) placed exactly one order.
- 2,997 people (3.1%) are repeat customers. They placed 6,342 orders (6.4% of all orders).
- The maximum is 17 orders by one person.

#### Geography: states and cities
**Question:** where do customers live, and how concentrated is it?

In [6]:
states = customers["customer_state"].value_counts()
state_tbl = pd.DataFrame({"n_customers": states,
                          "pct": (states / len(customers) * 100).round(1),
                          "cum_pct": (states.cumsum() / len(customers) * 100).round(1)})
print("states:", len(states))
display(state_tbl.head(8))
print("4 smallest states")
display(state_tbl.tail(4))

cities = customers["customer_city"].value_counts()
print("distinct cities                  :", len(cities))
print("cities with a single customer    :", (cities == 1).sum(), f"({(cities == 1).mean() * 100:.1f}% of cities)")
print("top 10 cities, share of customers:", round(cities.head(10).sum() / len(customers) * 100, 1), "%")
cities.head(5).rename("n_customers").to_frame()

states: 27


,n_customers,pct,cum_pct
customer_state,,,
SP,41746,42.0,42.0
RJ,12852,12.9,54.9
MG,11635,11.7,66.6
RS,5466,5.5,72.1
PR,5045,5.1,77.2
SC,3637,3.7,80.8
BA,3380,3.4,84.2
DF,2140,2.2,86.4


4 smallest states


,n_customers,pct,cum_pct
customer_state,,,
AM,148,0.1,99.8
AC,81,0.1,99.9
AP,68,0.1,100.0
RR,46,0.0,100.0


distinct cities                  : 4119
cities with a single customer    : 1144 (27.8% of cities)
top 10 cities, share of customers: 35.2 %


,n_customers
customer_city,
sao paulo,15540
rio de janeiro,6882
belo horizonte,2773
brasilia,2131
curitiba,1521


- 27 states appear. SP alone has 41,746 customers (42.0%), and the top 3 states (SP, RJ, MG) cover 66.6%.
- 4,119 distinct cities. The top 10 cities hold 35.2% of customers, and São Paulo city alone has 15,540.
- 1,144 cities (27.8%) have a single customer. The tail of cities is long and thin.

#### Same person, different address
**Question:** does a repeat customer always keep the same address?

In [7]:
people = customers.groupby("customer_unique_id").agg(n_orders=("customer_id", "size"),
                                                     n_zip=("customer_zip_code_prefix", "nunique"),
                                                     n_state=("customer_state", "nunique"))
repeat_people = people[people["n_orders"] > 1]

print("repeat people                  :", len(repeat_people))
print("... with more than one zip     :", (repeat_people["n_zip"] > 1).sum())
print("... with more than one state   :", (repeat_people["n_state"] > 1).sum())

repeat people                  : 2997
... with more than one zip     : 250
... with more than one state   : 39


- Of 2,997 repeat people, 250 used more than one zip prefix and 39 used more than one state.
- So an address belongs to the order, not to the person: customers move, or ship to someone else.